# H1. Cumulative Quantization Error in the Autoregressive KV Cache

*Status: scaffold. This article is written after the runs finish and reports them. It never loads a checkpoint and never produces a number that lands in `results/` — heavy execution belongs to `scripts/` under a `mise` task (see `docs/pmkvq-h1-todo.md`).*

The claim under examination: does per-step KV-cache quantization error **accumulate** with decode position? Three outcomes are possible a priori — growth, saturation, decay — and the naive reading of "accumulation" assumes the first without argument. We convert it into a growth law and measure the exponent.

## Helpers

`show_source`, `run_test`, `show_image`, `show_gif`, `REPO_ROOT`, `ASSETS`. Same shape as the reference repo.

In [ ]:
import sys, inspect, subprocess, base64
from pathlib import Path
from IPython.display import Image, display, Markdown

REPO_ROOT = Path.cwd()
ASSETS = REPO_ROOT / "assets"
sys.path.insert(0, str(REPO_ROOT))


def show_source(obj):
    """Render the real definition so the page cannot drift from the module."""
    display(Markdown("```python\n" + inspect.getsource(obj) + "\n```"))


def run_test(path):
    """A passing test prints nothing and returns 0. Any output is a failure."""
    r = subprocess.run([sys.executable, str(REPO_ROOT / path)],
                       capture_output=True, text=True)
    if r.returncode != 0 or r.stdout.strip():
        print(r.stdout, r.stderr)


def show_image(name):
    display(Image(filename=str(ASSETS / name)))


def show_gif(name):
    data = base64.b64encode((ASSETS / name).read_bytes()).decode()
    display(Markdown(f"![{name}](data:image/gif;base64,{data})"))


## 1. A quantizer is a noise injection with a known floor

Definition 1 and Lemma 1. The noise floor is $\sigma^2(b) = R^2 / 12(2^b-1)^2$. Moving 16-bit → 2-bit multiplies the noise power by ≈ $4.77\times10^8$ (≈ 86.8 dB) — the static stakes any temporal effect is weighed against.

In [ ]:
from pmkvq import quantizer
show_source(quantizer.fake_quant_group)

In [ ]:
run_test("tests/test_quantizer_floor.py")

*Asset A1 — `h1_noise_floor.png` (synthetic).*

In [ ]:
show_image("h1_noise_floor.png")

## 2. Where the error enters the readout

Proposition 1: the output perturbation splits into a value path $\sum_t a_{T,t}\delta^V_t$ and a mean-centred key path $\sum_t a_{T,t}(\Delta z_{T,t}-\overline{\Delta z}_T)v_t$. The centring is the formal statement of why the Key cache is the more sensitive tensor.

*Asset A2 — `h1_softmax_paths.png` (synthetic).*

In [ ]:
show_image("h1_softmax_paths.png")

## 3. Averaging suppresses independent noise

Lemma 3 and Corollary 2. The readout energy is $d\sigma^2[(1-\rho)/T_{\mathrm{eff}} + \rho]$. Neither $\rho=0$ (decaying) nor $\rho=1$ (flat) grows. **Growth requires a mechanism outside the readout.**

*Asset A3 — `h1_averaging.png` (synthetic).*

In [ ]:
show_image("h1_averaging.png")

## 4. Feedback is the only growth mechanism

Model 1 and Theorem 1. The autoregressive loop gives the linear ODE $dS/dT = c(T) + (\gamma^2/T)S$, whose closed form is $\varepsilon(T) = c/(1-\gamma^2) + A\gamma^2 T^{\alpha}$, $\alpha = \gamma^2 - 1$. A log-log plot has slope $\alpha$; the gain is $\gamma=\sqrt{\alpha+1}$.

In [ ]:
run_test("tests/test_fit_recovery.py")

*Asset A4 — `h1_regimes.png` (synthetic).*

In [ ]:
show_image("h1_regimes.png")

## 5. The measurement — Arm A against Arm B

The feedback switch made visible in the control flow. Only this section and Section 6 may state measured numbers.

In [ ]:
from pmkvq import run_experiment
show_source(run_experiment.decode_loop)

*Assets A5–A7 — `h1_loglog.png`, `h1_residuals.png`, `h1_teff.png` (measured). Populated after `mise run h1`.*

In [ ]:
# show_image("h1_loglog.png")   # uncomment once results exist

## 6. Early tokens dominate — Theorem 2 and Arm C

The Green's function $G(T,t_0) = (c_0\gamma^2/t_0)(T/t_0)^{\alpha}$ is monotonically decreasing in $t_0$ for every $\gamma^2>0$. This is the structural argument for progressive quantization, robust to the sign of $\alpha$.

*Assets A8, A9, A12 — measured.*

## 7. The optimal schedule — Theorem 3

$b^*(t) = b_0 - \gamma^2\log_4 t$ against the reference staircase $16\to8\to4\to2$. The mismatch is an open check.

*Asset A10 — `h1_bitschedule.png` (synthetic).*

In [ ]:
show_image("h1_bitschedule.png")

## 8. What this does not say, and the ESCP bridge

Scope boundary from thinkbook Section 13, then the per-byte cost bridge of Section 12 with placeholder $E_{\mathrm{byte}}$: $b^* = \log_4(2\lambda R^2\ln 4 / 3 n E_{\mathrm{byte}})$. Energy is linear in $b$, distortion exponential, so the knee is well defined.

*Asset A11 — `h1_bitsweep.png` (measured).*